# Time Series Volatility
**Área:** 04-econometrics. **Datos:** sintéticos, semilla 42. **Propósito:** laboratorio ejecutable de identificación y sensibilidad, no evidencia empírica del fenómeno.

## Ecuación principal
\[sigma_t^2=omega+alpha eps_(t-1)^2+beta sigma_(t-1)^2\]

**Pregunta:** Riesgo macrofinanciero. Se usa un DGP lineal como baseline metodológico; la ecuación económica del proyecto requiere calibración específica antes de cualquier conclusión aplicada.


In [ ]:
from pathlib import Path
import sys, math
root = Path.cwd()
for parent in [root, *root.parents]:
    candidate = parent / "01-economics-and-econometrics" / "computational-economics" / "src"
    if candidate.exists():
        sys.path.insert(0, str(candidate)); break
else:
    raise FileNotFoundError("Ejecutar notebook desde el repositorio o sus subdirectorios")
from econlab import generate_panel, ols, predict, rmse, bootstrap_slope
x, y = generate_panel(n=240, seed=42, slope=0.8, noise=0.3)
assert len(x)==len(y)==240
print("n=", len(x), "media y=", round(sum(y)/len(y),4))


## Estimación y validación temporal
Partición secuencial 70/30; en datos reales debe respetarse la fecha de disponibilidad (vintage), revisiones y unidad de análisis. La regresión simple no resuelve endogeneidad.


In [ ]:
split = int(.7*len(x))
coef = ols(x[:split],y[:split])
baseline = [sum(y[:split])/split]*len(y[split:])
test_pred = predict(coef,x[split:])
print("intercepto, pendiente:", coef)
print("RMSE modelo:",rmse(y[split:],test_pred),"RMSE baseline:",rmse(y[split:],baseline))
print("Bootstrap 95% (entrenamiento):",bootstrap_slope(x[:split],y[:split],replications=300,seed=42))
assert math.isfinite(coef[1]) and rmse(y[split:],test_pred)>=0


## Robustez y análisis de sensibilidad
Comparar cambios de signo, escala, ruido y tamaño muestral. La tasa de cobertura del bootstrap requiere replicaciones Monte Carlo independientes; el intervalo percentil puede fallar con dependencia temporal o muestras pequeñas.


In [ ]:
for noise in (0.1,0.3,0.8,1.5):
    xx,yy=generate_panel(n=240,seed=42,noise=noise)
    b=ols(xx,yy)[1]
    print(f"ruido={noise:.1f} pendiente={b:.4f} error={abs(b-0.8):.4f}")
assert abs(ols(*generate_panel(n=240,seed=42,noise=0))[1]-.8)<1e-10


## Interpretación y extensión de investigación
- **Ecuación del proyecto:** sigma_t^2=omega+alpha eps_(t-1)^2+beta sigma_(t-1)^2.
- **Identificación:** especificar exogeneidad, instrumento, experimento o diseño estructural; no atribuir causalidad a OLS.
- **Fintech:** Riesgo macrofinanciero.
- **Siguiente paso:** sustituir el DGP genérico por un generador estructural, agregar datos públicos con fecha de corte y contrastar modelos alternativos.
- **Tests:** ejecutar `python -m unittest discover -s 01-economics-and-econometrics/computational-economics/tests -v`.
